<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c07-header.png" alt="Nextia Learning · LLMs: How They Work and How to Choose One" width="100%">

# Solution: From text to tokens

**[C07-M01-L01 · From text to tokens](https://learning.nextia-ai.com/courses/llms/m01/from-text-to-tokens/)** · C07, LLMs: How They Work and How to Choose One · Module 1, lesson 1 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** cut a text into tokens with a real tokenizer, read the token IDs, and measure how many tokens the same text needs in different languages.

| | |
|---|---|
| **Time** | About 20 minutes |
| **Needs** | An internet connection for the setup cell. No account and no API key. The setup installs tiktoken 0.14.0. |
| **You should know** | How to run a notebook cell, and what a Python list and a dictionary are (C02). The lesson page explains every idea. |
| **Optional** | The lesson makes complete sense without this notebook. Use it to see the numbers come from real code. |
| **Tested** | 2026-10-08, Python 3.14.6 with tiktoken 0.14.0, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/llms/m01/from-text-to-tokens/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C07/M01-L01-from-text-to-tokens/from-text-to-tokens-solution.ipynb).

## Setup

The next cell installs tiktoken==0.14.0, downloads the course data from the [labs repository](https://github.com/usmanahmed99/nextia-learning-labs/tree/main/C07/data) and checks each file. It also defines `expect()` and `expect_hash()`, which check your answers without showing them.

> **Check.** The last line starts with `Ready:`. If you see a checksum error, delete the folder `c07-data` and run the cell again.

In [ ]:
# Setup: packages and data. Run this cell first.
import hashlib, json, subprocess, sys, urllib.request
from pathlib import Path
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--disable-pip-version-check', 'tiktoken==0.14.0'], check=True)

DATA_URL = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C07/data/"
FILES = {'parallel.json': 'b5b58f38361a35f83f289048949eee6caedc49fb7283cfc967d2c499b152adf1', 'tickets.jsonl': '2804678eb3fed7d0e925398daeaa8632d83e1df12bc88e984c38988b20fc86bf'}
DATA = Path("c07-data")
DATA.mkdir(exist_ok=True)
for name, expected in FILES.items():
    path = DATA / name
    if not path.exists() or hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        try:
            urllib.request.urlretrieve(DATA_URL + name, path)
        except OSError:
            subprocess.run(["curl", "-fsSL", "-o", str(path), DATA_URL + name], check=True)
    if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder c07-data and run again.")

def expect(what, yours, expected, tolerance=1e-6):
    """Compare your number with the expected one, without showing the answer."""
    if isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        if abs(yours - expected) <= tolerance:
            print(f"Check passed: {what} is right.")
        else:
            print(f"Not yet: {what} is too {'high' if yours > expected else 'low'}.")
    elif yours == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected value.")

def fingerprint(value):
    """A short hash of a value (numbers rounded to 2 decimals)."""
    norm = round(value, 2) if isinstance(value, float) else value
    return hashlib.sha256(repr(norm).encode()).hexdigest()[:12]

def expect_hash(what, yours, expected_hash):
    """Compare your answer with a hidden fingerprint of the expected one."""
    if fingerprint(yours) == expected_hash:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected value. Check the steps above.")

print("Ready:", ", ".join(FILES))

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Words and their tokens

A tokenizer cuts text into **tokens**, the pieces in its **vocabulary**, and turns each piece into its **token ID**. Run the cell to load two of OpenAI's tokenizers and cut eight words. Each word has a space before it, as in a sentence.

> **Predict.** Which of these words is one token in both tokenizers, and which are cut?

> **Check.** You should see ` refund` as one token in both, and ` remboursement` as 1 token in o200k_base but 4 in cl100k_base.

In [ ]:
import tiktoken

encodings = {name: tiktoken.get_encoding(name) for name in ["o200k_base", "cl100k_base"]}
words = [" refund", " remboursement", " ceramic", " céramique", " broken", " cassé", " Larkfield", " L-20431"]
for name, enc in encodings.items():
    print(f"{name}: {enc.n_vocab} tokens in the vocabulary")
    for word in words:
        ids = enc.encode(word)
        print(f"  {word!r:18} {len(ids)} token(s): {[enc.decode([i]) for i in ids]}  IDs {ids}")

Notice that the same piece has a different ID in each vocabulary: ` refund` is 18376 in o200k_base and 21639 in cl100k_base. A model always uses the tokenizer it was trained with.

## 2. One sentence in eight languages

`parallel.json` holds one support sentence in eight languages. The sentence is synthetic, written for the course; the translations other than French are machine-assisted and checked for French only.

> **Predict.** The English sentence has 20 tokens with o200k_base. Which language needs the most tokens with cl100k_base?

In [ ]:
sentences = json.loads((DATA / "parallel.json").read_text(encoding="utf-8"))["sentences"]
print(f"{'language':10}{'characters':>12}{'o200k_base':>12}{'cl100k_base':>13}")
for lang, text in sentences.items():
    counts = [len(enc.encode(text)) for enc in encodings.values()]
    print(f"{lang:10}{len(text):>12}{counts[0]:>12}{counts[1]:>13}")

> **Check.** English is 20 and 20. French is 26 and 34. Hindi is 36 and 111: the older tokenizer has no piece for most Hindi characters, so it spells them in raw bytes.

## 3. Your turn: count one ticket

The tickets of the course are in `tickets.jsonl`, one JSON object per line. Ticket S07 is a French ticket about a late order.

> **Your turn.** Count the tokens of the text of ticket S07 with o200k_base. Put the count in `s07_tokens`.

In [ ]:
tickets = [json.loads(line) for line in (DATA / "tickets.jsonl").read_text(encoding="utf-8").splitlines()]
by_id = {t["id"]: t for t in tickets}
o200k = encodings["o200k_base"]

s07_tokens = len(o200k.encode(by_id["S07"]["text"]))
print(s07_tokens)

Run the check.

In [ ]:
expect_hash("s07_tokens", s07_tokens, "eb1e33e8a81b")

## 4. Change one thing: tokens per word, English against French

The twelve routine tickets (S01 to S12) are six in English and six in French. Compare how many tokens each language needs **per word**.

> **Your turn.** Complete `tokens_per_word(lang)`: add up the tokens and the words of the routine tickets in that language (`t["expected"]["language"]`), and return tokens divided by words.

In [ ]:
def tokens_per_word(lang):
    routine = [t for t in tickets if t["id"].startswith("S") and t["expected"]["language"] == lang]
    tokens = sum(len(o200k.encode(t["text"])) for t in routine)
    words = sum(len(t["text"].split()) for t in routine)
    return tokens / words

print("English:", tokens_per_word("en"))
print("French:", tokens_per_word("fr"))

Run the check. It compares both numbers, rounded to 2 decimals.

In [ ]:
expect_hash("tokens_per_word('en')", tokens_per_word("en"), "c63c003345e8")
expect_hash("tokens_per_word('fr')", tokens_per_word("fr"), "a8aaebadf1fb")

> **Check.** French needs more tokens per word than English. Write one sentence for Omar: what does this mean for the cost of a French ticket?

## 5. Failure case: counting words instead of tokens

A budget that counts words is too low. Run the cell: it compares the words and the tokens of every ticket.

In [ ]:
words = sum(len(t["text"].split()) for t in tickets)
tokens = sum(len(o200k.encode(t["text"])) for t in tickets)
print(f"24 tickets: {words} words, {tokens} tokens ({tokens / words:.2f} tokens per word)")

> **Check.** There are more tokens than words. Plan budgets and limits in tokens, with the tokenizer of the model you will use, and with a sample of every language your customers write in.

## Recap

- A tokenizer turns text into token IDs; `encode` and `decode` go both ways.
- The same sentence needs a different number of tokens in each language and with each tokenizer.
- French tickets need more tokens per word than English ones, so they cost more and fill more of the model's room.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Predicting continuations](https://learning.nextia-ai.com/courses/llms/m01/predicting-continuations/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/llms/m01/from-text-to-tokens/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.